
<p style="font-size:50px">
<span style="font-size:200px">
&#9703;
</span>
Faire des expériences
</p>

---



Sous-sections :
[Les jeux de données](#datasets)&nbsp;| 
[La base de cas](#cb)&nbsp;| 
[Le nouveau cas](#ct)&nbsp;| 
[La similarité](#sim)&nbsp;| 
[On met tout ensemble](#tout)&nbsp;| 



Le but de cette séance est d'utiliser la mini-librairie que nous avons constitué pour lancer des expériences sur des données.
On veut pouvoir facilement appliquer l'algorithme kNN sur des données en faisant varier différents paramètres, comme :
- le paramètre $k$
- la mesure de similarité
- le jeu de données

<a id="datasets"></a>
### Les jeux de données

__Breast Cancer Wisconsin__

Le jeu de données [Breast Cancer Wisconsin](https://archive.ics.uci.edu/dataset/15/breast+cancer+wisconsin+original) contient 699 instances décrivant des patients atteints du cancer du sein.

Dans ce jeu de données, 
chaque patient est décrit par ses résultats de tests de cytopathologie, et sa tumeur est classifiée dans l'une des deux classes `2` (Benign) ou `4` (Malignant).

Le jeu de données est souvent utilisé pour tester des algorithmes de classification, dans lesquels la tâche consiste à prédire la classe d'un nouveau patient à partir de sa description.

Le fichier `breast-cancer-wisconsin.data` contient les données au format csv.
La première colonne est un identifiant unique, les 9 suivantes sont des attributs descriptifs de la cytologie du patient, et le dernier est la classe.  

Pour ce jeu de données, on voudrait pouvoir charger et utiliser le jeu de données de la manière suivante :

In [ ]:
import cbp

breast = cbp.load_dataset('breast')
breast.X, breast.y, len(breast), breast[:8]

__CT Images__ 

Nous travaillons ici sur des images de radiologie : https://www.kaggle.com/datasets/kmader/siim-medical-images/data

L'échantillon contient $100$ images, pour lesquelles on peut associer un label :
- $50$ images ont le label $0$ (normal, pas de cancer sur l'image)
- $50$ images ont le label $1$ (cancereux, présence d'un cancer sur l'image)

In [ ]:
import cbp
ct_images = cbp.load_dataset('ct_images')
ct_images.X, ct_images.y

In [ ]:
ct_images.as_images().afficher_images([29,10,5])

In [ ]:
ct_images.as_vectors().X[0]

__Lung Cancer__ 

Ce jeu de données contient $1097$ images de radiographie des poumons :
https://www.kaggle.com/datasets/leelanaveenkumar/lung-cancer-ct-slice-images-metadatasynthetic

Chaque image est accompagnée d'un label :
- $536$ images ont le label $0$ (normal, pas de cancer sur l'image)
- $561$ images ont le label $1$ (cancereux, présence d'un cancer sur l'image)

Pour ce jeu de données, on voudrait pouvoir charger et utiliser le jeu de données de la manière suivante : 

In [ ]:
import cbp
lung = cbp.load_dataset('lung')
lung.X, lung.y

In [ ]:
lung.as_images().afficher_images([29,10,5])

<a id="cb"></a>
### La base de cas

Pour choisir une base de cas, on filtre le jeu de données par un ensemble d'indices :

In [ ]:
cb_indices = [9, 13, 29, 30, 45]
X_cb, y_cb = ct_images[cb_indices]


<a id="ct"></a>
### Le cas cible

Pour choisir un cas cible, on fait de même mais en donnant un seul indice :

In [ ]:
cible_idx = 79
x, y_true = ct_images.as_vectors()[cible_idx]

<a id="sim"></a>
### La similarité

On choisit une mesure de similarité :

In [ ]:
from cbp.similarity.measures import cosin

sim = cosin

Par exemple :

In [ ]:
from cbp.algo.knn import kneighbors

import numpy as np
X = ct_images.as_vectors()
X_cb = np.concatenate((X[:cible_idx][0],X[cible_idx+1:][0]))
ct_images.as_images().afficher_images([cible_idx]+list(kneighbors(x, X_cb, sim, n_neighbors=7)))

<a id="tout"></a>
### On met tout ensemble

Dans cette expérience, on prend aléatoirement une partie du jeu de données pour servir de base de cas et on calcule le taux d'erreur de l'algorithme des $k$ plus proches voisins sur les autres instances. 

Sur Breast Cancer Wisconsin avec la distance Euclidienne :

In [ ]:
from cbp.similarity.measures import euclidean_sim
from cbp.algo import knn

# the similarity measure
sim = euclidean_sim

# the value of k
k = 3

data = breast

# Randomly split the dataset with reproducibility
from sklearn.model_selection import train_test_split
X_cb, X_test, y_cb, y_test = train_test_split(data.X, data.y, train_size=.8, stratify=data.y, random_state=34)

acc = 0.
for x, y_true in zip(X_test,y_test):
    y_pred = knn.predict(x, X_cb, y_cb, sim, n_neighbors=k)
    print(f'y_true={y_true}, y_pred={y_pred}')
    if y_pred == y_true:
        acc += 1
acc /= len(X_test)

print(f'La performance des {k}-NN sur {data} est {acc:.4f}%')


Sur CT Images avec la similarité cosinus :

In [ ]:
from cbp.similarity.measures import cosin
from cbp.algo import knn

# the similarity measure
sim = cosin

# the value of k
k = 7

data = ct_images.as_vectors()

# Randomly split the dataset with reproducibility
from sklearn.model_selection import train_test_split
X_cb, X_test, y_cb, y_test = train_test_split(data.X, data.y, train_size=50, test_size=50, stratify=data.y, random_state=34)

acc = 0.
for x, y_true in zip(X_test,y_test):
    y_pred = knn.predict(x, X_cb, y_cb, sim, n_neighbors=k)
    print(f'y_true={y_true}, y_pred={y_pred} ')
    if y_pred == y_true:
        acc += 1
acc /= len(X_test)

print(f'La performance des {k}-NN sur {data} est {acc:.4f}%')
